## Random Forest

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Load dataset 
df = pd.read_csv("liquid_audio_features_filtered.csv")
print(df.head())

# 2. Split features and labels 
X = df.drop("label", axis=1).values
y = df["label"].values

# 3. Encode labels 
le = LabelEncoder()
y_encoded = le.fit_transform(y)
print("Label mapping:", dict(zip(le.classes_, le.transform(le.classes_))))

# 4. Train/Test Split 
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

# 5. Scale features --> Optional
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# 6. Train Classifier 
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)

# 7. Evaluate 
y_pred = clf.predict(X_test)
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred, target_names=le.classes_))

# 8. Confusion Matrix 
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", xticklabels=le.classes_, yticklabels=le.classes_, cmap="Blues")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Random Forest Confusion Matrix")
plt.show()

# 8. Bar Plot of Classification Metrics 
metrics = ['precision', 'recall', 'f1-score']
classes = le.classes_

# 10. Create a DataFrame 
# allows for easy storage and plot of classification metric data

# classification report as a dictionary
report = classification_report(y_test, y_pred, target_names=le.classes_, output_dict=True)

# extract data for desired metrics
data = [[report[label][metric] for metric in metrics] for label in classes]
df_metrics = pd.DataFrame(data, index=classes)  #DATA VARAIBLE NOT DEFINED, NOT WORKING

# 11. Plot the bar chart 
df_metrics.plot(kind='bar', figsize=(10, 6), colormap='Set2')
plt.title("Random Forest Classification Metrics by Liquid Type")
plt.xlabel("Liquid Type")
plt.xticks(rotation=0)
plt.ylabel("Score")
plt.tight_layout()
plt.legend(title="Metrics", loc='lower left')
plt.show()

ModuleNotFoundError: No module named 'pandas'

### RF Feature Importance

In [ ]:
# 1. Retrieve feature names from original DataFrame (excluding 'label')
feature_names = df.drop("label", axis=1).columns

# 2. Retrieve feature importances from trained classifier
importances = clf.feature_importances_

# 3. Create a DataFrame for visualization
feature_importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

# 4. Print feature importance table
print("\nFeature Importance:\n")
print(feature_importance_df)

# 5. Plot results
plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=feature_importance_df, palette='viridis')
plt.title('Feature Importance - Random Forest')
plt.xlabel('Importance Score')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

### Experiment with Most Influential Features

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score

# 1. Top features 
top_n = len(feature_names)  # will check from 1 - 17 to gauge best variations

# Store F1-scores for the different subsets
f1_scores = []

# 2. Train and evaluate model for different feature subsets
for n in range(1, top_n + 1):
    # Select the most important features (loop helps identify top #)
    top_features = feature_importance_df['Feature'][:n]
    X_subset = df[top_features].values
    y = df['label'].values

    # Repeat steps of RF model for each subset

    # Split data into training and testing sets
    X_train, X_test, y_train, y_test = train_test_split(X_subset, y, test_size=0.2, random_state=42, stratify=y)
    
    # Standardize features
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)
    
    # Train Random Forest
    clf = RandomForestClassifier(n_estimators=100, random_state=42)
    clf.fit(X_train, y_train)
    
    # Predict and calculate F1-score
    y_pred = clf.predict(X_test)
    f1 = f1_score(y_test, y_pred, average='weighted')  # 'weighted' for class imbalance
    f1_scores.append(f1)

# 3. List the F1-scores for the different feature subsets
print("F1-Scores for Different Feature Subsets:")
for n, f1 in zip(range(1, top_n + 1), f1_scores):
    print(f"Top {n} Features: F1-Score = {f1:.4f}")

# 4. Plot F1-scores for different feature subsets
plt.figure(figsize=(10, 6))
plt.plot(range(1, top_n + 1), f1_scores, marker='o')
plt.title('F1-Score for Different Feature Subsets')
plt.xlabel('Number of Top Features')
plt.ylabel('F1-Score')
plt.xticks(range(1, top_n + 1))
plt.tight_layout()
plt.show()

### Overfitting Analysis

In [ ]:
import matplotlib.pyplot as plt
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# 1. Predict and evaluate RF model performance
train_preds = clf.predict(X_train)
test_preds = clf.predict(X_test)

train_accuracy = accuracy_score(y_train, train_preds)
test_accuracy = accuracy_score(y_test, test_preds)

print(f"Training Accuracy: {train_accuracy:.4f}")
print(f"Testing Accuracy: {test_accuracy:.4f}")

# 2. Check for gap between training and testing accuracy
if train_accuracy - test_accuracy > 0.1:
    print("Warning: Potential overfitting detected")

# 3. Cross-Validate Model Performance Scores
cross_val_scores = cross_val_score(clf, X_subset, y, cv=5)
print(f"Cross-validation Scores: {cross_val_scores}")
print(f"Mean Cross-validation Score: {cross_val_scores.mean():.4f}")

# 4. Learning Curves - Plot Training vs Testing Score
train_sizes = range(1, len(X_train), len(X_train)//10)
train_scores, test_scores = [], []

for size in train_sizes:
    X_train_subset = X_train[:size]
    y_train_subset = y_train[:size]
    clf.fit(X_train_subset, y_train_subset)
    
    train_scores.append(clf.score(X_train_subset, y_train_subset))
    test_scores.append(clf.score(X_test, y_test))

plt.figure(figsize=(10, 6))
plt.plot(train_sizes, train_scores, label='Train Score')
plt.plot(train_sizes, test_scores, label='Test Score')
plt.xlabel('Training Set Size')
plt.ylabel('Score')
plt.title('Learning Curve') # Training vs Testing Score
plt.legend()
plt.show()

# 5. Model Complexity - Experimenting with different n_estimators and max_depth values
train_accuracies, test_accuracies = [], []

for n_estimators in [10, 50, 100, 200]:
    for max_depth in [5, 10, 20, None]:
        clf = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
        clf.fit(X_train, y_train)
        
        train_accuracies.append(clf.score(X_train, y_train))
        test_accuracies.append(clf.score(X_test, y_test))

# 6. Plot the accuracies
plt.figure(figsize=(10, 6))
plt.plot(train_accuracies, label='Train Accuracy')
plt.plot(test_accuracies, label='Test Accuracy')
plt.xlabel('Model Complexity (n_estimators and max_depth)')
plt.ylabel('Accuracy')
plt.title('Accuracy vs Model Complexity') # Overfitting Check
plt.legend()
plt.show()


#### Cross Validation Learning Curve - RF

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Cross Validation Learning Curve for Random Forest 

# 1. Setup Hyperparameter Grid
rf_params = { # reduced because initial grid attempt was too large
    'n_estimators': [100, 200, 400],  # num of trees in the forest
    'max_depth': [None, 10, 20],  # max depth of the trees
    'min_samples_split': [2, 5],  # min samples req to split a node
    'min_samples_leaf': [1, 2],  # min samples req to be at a leaf node
    'max_features': ['sqrt', 'log2', None]  # num of features to consider at each split
}

# 2. Initialize
rf = RandomForestClassifier(random_state=42)
# rf_grid_search = GridSearchCV(rf, rf_params, cv=5, scoring='accuracy', verbose=1)

rf_grid_search = GridSearchCV(rf, rf_params, cv=5, n_jobs=-1)
# grid_search.fit(X_train, y_train)

# 3. Fit the model to the training data
rf_grid_search.fit(X_train, y_train)

# 4. Get the best parameters 
print("Best Random Forest Parameters:", rf_grid_search.best_params_)

# 5. Evaluate the best model 
y_pred_rf = rf_grid_search.predict(X_test)
print("Random Forest Classification Report:")
print(classification_report(y_test, y_pred_rf, target_names=le.classes_))

# 6. Identify the Best Parameters + Gauge Accuracy 
print(f"Best Random Forest Parameters: {rf_grid_search.best_params_}")
best_rf = rf_grid_search.best_estimator_
train_preds = best_rf.predict(X_train)
test_preds = best_rf.predict(X_test)

train_accuracy = accuracy_score(y_train, train_preds)
test_accuracy = accuracy_score(y_test, test_preds)

print(f"Training Accuracy: {train_accuracy:.4f}")
print(f"Testing Accuracy: {test_accuracy:.4f}")

# 7. Confusion Matrix 
cm_rf = confusion_matrix(y_test, y_pred_rf)

# 8. Plot the Confusion Matrix 
plt.figure(figsize=(8, 6))
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Blues', xticklabels=le.classes_, yticklabels=le.classes_)
plt.title("Random Forest Confusion Matrix")
plt.xlabel('Predicted Labels')
plt.ylabel('True Labels')
plt.show()

## MLP 

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Load feature dataset 
df = pd.read_csv("liquid_audio_features_filtered.csv")
X = df.drop("label", axis=1).values
y = df["label"].values

# 2. Encode labels 
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# 3. Train/test split 
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

# 4. Feature scaling 
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# 5. MLP Model 
mlp = MLPClassifier(
    hidden_layer_sizes=(64, 32),  # Can experiment with this!
    activation="relu",
    solver="adam",
    max_iter=300,
    random_state=42
)

mlp.fit(X_train, y_train)

# 6. Evaluation 
y_pred = mlp.predict(X_test)

print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=le.classes_))

# 7. Confusion matrix 
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", xticklabels=le.classes_, yticklabels=le.classes_, cmap="Blues")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("MLP Confusion Matrix")
plt.show()


ModuleNotFoundError: No module named 'sklearn'

## KNN

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report

# 1. Load feature dataset 
df = pd.read_csv("liquid_audio_features_filtered.csv")
X = df.drop("label", axis=1).values
y = df["label"].values

# 2. Train & evaluate KNN 
knn = KNeighborsClassifier(n_neighbors=10) 
knn.fit(X_train, y_train) # initalize the KNN model
y_pred_knn = knn.predict(X_test)

print("KNN Classification Report: ")
print(classification_report(y_test, y_pred_knn, target_names=le.classes_))

# 3. Dot Plot 
plt.figure(figsize=(10, 6))

# true labels displayed as dots
plt.scatter(range(len(y_test)), y_test, color='cyan', label='True', alpha=0.6)

# predicted labels as x's because our accuracy is very good :D
plt.scatter(range(len(y_pred)), y_pred, color='magenta', marker='x', label='Predicted', alpha=0.6)

plt.title("KNN Classifier - Dot Plot: Predicted vs True Labels")
plt.xlabel("Sample Index")
plt.ylabel("Liquid")
plt.legend()
plt.grid(True)
plt.yticks(ticks=range(len(le.classes_)), labels=le.classes_)  # y-axis liquid label names
plt.tight_layout()
plt.show()

# 4. Confusion Matrix =

cm = confusion_matrix(y_test, y_pred_knn)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=le.classes_, yticklabels=le.classes_)
plt.title('KNN Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.show()

## SVM

In [ ]:
import pandas as pd
import numpy as np
from sklearn.svm import SVC
from sklearn.decomposition import PCA
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

# 1. Load feature dataset 
df = pd.read_csv("liquid_audio_features_filtered.csv")
X = df.drop("label", axis=1).values
y = df["label"].values

# 2. Encode labels 
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# 3. Train/Test Split 
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

# 4. Scale Features 
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# 5. PCA for visualization 
# reduces to 2D so we can actually visualize the data
pca = PCA(n_components=2)
X_test_2D = pca.fit_transform(X_test)

# 6. Train SVM on original data 
svm = SVC(kernel='rbf', C=1, gamma='scale')
svm.fit(X_train, y_train)

# 7. Predict and Evaluation 
y_pred = svm.predict(X_test)

print("SVM Classification Report:")
print(classification_report(y_test, y_pred, target_names=le.classes_))

# 8. Create mesh grid for plotting decision boundary 
h = .02  # smaller step size in the mesh grid to plot more points
x_min, x_max = X_test_2D[:, 0].min() - 1, X_test_2D[:, 0].max() + 1
y_min, y_max = X_test_2D[:, 1].min() - 1, X_test_2D[:, 1].max() + 1

xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                     np.arange(y_min, y_max, h))


# 9. Predict the classification for each point in the mesh grid 
Z = svm.predict(pca.inverse_transform(np.c_[xx.ravel(), yy.ravel()]))
Z = Z.reshape(xx.shape)

# 10. Plot true and predict side-by-side with decision boundary 
plt.figure(figsize=(12, 5))

# 11. True labels with decision boundary 
plt.subplot(1, 2, 1)
plt.contourf(xx, yy, Z, alpha=0.3, cmap=plt.cm.coolwarm)
for label in np.unique(y_test):
    idxs = np.where(y_test == label)
    plt.scatter(X_test_2D[idxs, 0], X_test_2D[idxs, 1], label=le.classes_[label], alpha=0.7)
plt.title("True Labels with SVM Decision Boundary")
plt.xlabel("PCA Component 1")
plt.ylabel("PCA Component 2")
plt.legend()

# 12. Predicted labels with decision boundary 
plt.subplot(1, 2, 2)
plt.contourf(xx, yy, Z, alpha=0.3, cmap=plt.cm.coolwarm)
for label in np.unique(y_pred):
    idxs = np.where(y_pred == label)
    plt.scatter(X_test_2D[idxs, 0], X_test_2D[idxs, 1], label=le.classes_[label], alpha=0.7)
plt.title("SVM Predicted Labels with Decision Boundary")
plt.xlabel("PCA Component 1")
plt.ylabel("PCA Component 2")
plt.legend()

plt.tight_layout()
plt.show()

# 13. Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=le.classes_, yticklabels=le.classes_)
plt.title('SVM Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.show()